# Naive RAG — *Os Sertões*

Nesta abordagem, o livro é dividido em trechos menores. Esses trechos são transformados em embeddings, armazenados no ChromaDB e recuperados por similaridade para responder às perguntas.

## 1. Importações e chave da OpenAI

A chave é solicitada somente durante a execução e não fica salva no notebook. A API precisa ter créditos disponíveis.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains.question_answering import load_qa_chain
from langchain_core.prompts import ChatPromptTemplate

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / '.env')

if not os.environ.get('OPENAI_API_KEY'):
    raise ValueError('A chave OPENAI_API_KEY não foi encontrada no arquivo .env.')

## 2. Carregamento e divisão do PDF

In [ ]:
PDF_PATH = PROJECT_ROOT / 'data' / 'os-sertoes.pdf'

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=['\n\n', '\n', '. ', ' ', '']
)

chunks = text_splitter.split_documents(documents)
print(f'Páginas carregadas: {len(documents)}')
print(f'Trechos gerados: {len(chunks)}')

## 3. Criação do banco vetorial

Esta célula gera embeddings do livro e pode consumir créditos da API. Depois de criado, o banco fica salvo localmente.

In [ ]:
PERSIST_DIRECTORY = str(PROJECT_ROOT / 'chroma_db' / 'naive_rag')

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIRECTORY
)
vectorstore.persist()

retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print('Banco vetorial criado com sucesso.')

## 4. Cadeia de perguntas e respostas

In [ ]:
TEMPLATE = """
Você é um assistente especializado na obra 'Os Sertões', de Euclides da Cunha.

Responda em português usando exclusivamente o contexto fornecido.
Se o contexto não for suficiente, informe claramente que não encontrou a informação.

Pergunta: {question}

Contexto:
{context}
"""

prompt = ChatPromptTemplate.from_template(TEMPLATE)
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
qa_chain = load_qa_chain(llm, chain_type='stuff', prompt=prompt)

def ask(question):
    context = retriever.get_relevant_documents(question)
    result = qa_chain.invoke({
        'input_documents': context,
        'question': question
    })
    return result['output_text'], context

## 5. Perguntas de avaliação

In [ ]:
questions = [
    'Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?',
    'Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?',
    'Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?',
    'Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?',
    'Quais são os principais aspectos da crítica social e política presentes em Os Sertões? Como esses aspectos refletem a visão do autor sobre o Brasil da época?'
]

for index, question in enumerate(questions, start=1):
    answer, context = ask(question)
    pages = sorted({document.metadata.get('page', 0) + 1 for document in context})
    print(f'Pergunta {index}: {question}')
    print(f'Resposta: {answer}')
    print(f'Páginas recuperadas: {pages}')
    print('-' * 100)